# Вариант 16. Задача 1. Линейное программирование
## Постановка задачи
На первом складе (А1) содержится сталь двух марок: 2900 т марки «а» и 3900 т марки «б».

Сталь этих марок содержится также и на втором складе (А2) в следующих количествах: 4800 т марки «а» и 3100 т марки «б».

Сталь должна быть вывезена в два пункта потребления: в пункт В1 необходимо поставить 2500 т стали марки «а», 3800 т марки «б» и остальные 400 т стали любой марки.

Аналогично второй пункт потребления В2 должен получить 5500 т стали, из них 2700 т марки «а» и 2400 т марки «б».

Известно, что 1 т стали марки «а» может заменить 0.7 т стали марки «б» (то есть, вместо стали марки «б» можно использовать соответствующее количество стали марки «а», но не наоборот).

Стоимость перевозок в денежных единицах (ДЕ) за тонну составляют: из пункта А1 в пункты В1 и В2 4.5 ДЕ и 2.1 ДЕ, и из пункта А2 в В1 и В2, соответственно, 5.9 ДЕ и 2.1 ДЕ.

Требуется найти план перевозок стали, минимизирующий затраты на перевозки; полученное решение необходимо исследовать.

1. Провести параметрическое изменение правых частей:
    - если в решении отсутствуют перевозки с заменой марок стали («а» на «б»), то провести постепенное изменение запасов на обоих складах (увеличение - для стали марки «а», и уменьшение для стали марки «б») до тех пор, пока по крайней мере 100 т стали марки «а» не будут направлены на замену стали марки «б»;
    - если в решении присутствуют такие перевозки, то провести обратное постепенное изменение запасов, пока перевозки с заменой марок стали не исчезнут (рассмотреть вариант только с увеличением запасов стали марки «б» без изменения запасов стали марки «а» на обоих складах).
2. Провести параметрическое изменение целевой функции:
    - предварительно существенно увеличить запасы стали обеих марок (в правых частях), чтобы на предпочтительность перевозок стали той или иной марки не влияли ограничения по запасам;
    - убедиться в отсутствии замен (или в их наличии), для чего заново решить задачу;
    - затем провести постепенное изменение (увеличение или уменьшение - по смыслу) стоимости перевозок стали марки «б» (при прежней стоимости перевозок стали марки «а») до появления хотя бы 100 т замены (или до исчезновения замен).

## Формализация задачи

Пусть $i \in \{1, 2\}$ (склады А1, А2) и $j \in \{1, 2\}$ (пункты В1, В2).

- $x_{i, j, a}$ — количество стали марки «а», перевозимой для удовлетворения основной потребности в марке «а» (т)
- $x_{i, j, b}$ — количество стали марки «б», перевозимой для удовлетворения основной потребности в марке «б» (т)
- $s_{i, j}$ — количество стали марки «а», направляемой на замену марки «б» (т)
- $z_{i, j, a}$ — количество стали марки «а», поставляемой как «любая марка» (т)
- $z_{i, j, b}$ — количество стали марки «б», поставляемой как «любая марка» (т)

Целевая функция: $\sum_{i=1}^{2} \sum_{j=1}^{2} C_{i,j} \cdot (x_{i,j,a} + x_{i,j,b} + s_{i,j} + z_{i,j,a} + z_{i,j,b}) \rightarrow \min$

Где стоимости $C_{i,j}$:

$C_{1,1} = 4.5$, $C_{1,2} = 2.1$

$C_{2,1} = 5.9$, $C_{2,2} = 2.1$


Ограничения по запасам на складах:

Склад А1 (марка «а»): $\sum_{j} (x_{1,j,a} + s_{1,j} + z_{1,j,a}) \leq 2900$

Склад А1 (марка «б»): $\sum_{j} (x_{1,j,b} + z_{1,j,b}) \leq 3900$

Склад А2 (марка «а»): $\sum_{j} (x_{2,j,a} + s_{2,j} + z_{2,j,a}) \leq 4800$

Склад А2 (марка «б»): $\sum_{j} (x_{2,j,b} + z_{2,j,b}) \leq 3100$

Ограничения по потребностям пунктов (с учетом замены):

Пункт В1:
- Потребность в «а»: $\sum_{i} x_{i,1,a} = 2500$
- Потребность в «б»: $\sum_{i} (x_{i,1,b} + 0.7 \cdot s_{i,1}) = 3800$
- Остаток (любая марка): $\sum_{i} (z_{i,1,a} + z_{i,1,b}) = 400$

Пункт В2:
- Потребность в «а»: $\sum_{i} x_{i,2,a} = 2700$
- Потребность в «б»: $\sum_{i} (x_{i,2,b} + 0.7 \cdot s_{i,2}) = 2400$
- Остаток (любая марка): $\sum_{i} (z_{i,2,a} + z_{i,2,b}) = 400$ (так как $5500 - 2700 - 2400 = 400$)

Условие неотрицательности: Все $x_{i,j,k}, s_{i,j}, z_{i,j,k} \geq 0$.

## Описание задачи с помощью средств библиотеки PuLP

In [1]:
import pulp

I = [1, 2]  # Индексы складов
J = [1, 2]  # Индексы пунктов потребления

# Стоимости перевозок C[i,j]
C_BASE = {(1,1): 4.5, (1,2): 2.1, (2,1): 5.9, (2,2): 2.1}

# Исходные запасы на складах
SUPPLY_A1_A, SUPPLY_A1_B = 2900, 3900
SUPPLY_A2_A, SUPPLY_A2_B = 4800, 3100

# Потребности пунктов
DEMAND = {
    (1, 'a'): 2500,   # В1 потребность в 'а'
    (1, 'b'): 3800,   # В1 потребность в 'б'
    (1, 'any'): 400,  # В1 потребность в "любой"
    (2, 'a'): 2700,   # В2 потребность в 'а'
    (2, 'b'): 2400,   # В2 потребность в 'б'
    (2, 'any'): 400   # В2 потребность в "любой"
}

SUBSTITUTION_RATE = 0.7  # 1 т "а" заменяет 0.7 т "б"

def create_and_solve_steel_problem(sup_a1, sup_b1, sup_a2, sup_b2, delta_cost_b=0.0, solve=True):
    p = pulp.LpProblem('SteelTransport', pulp.LpMinimize)

    x_a = pulp.LpVariable.dicts('x_a', [(i, j) for i in I for j in J], lowBound=0)
    x_b = pulp.LpVariable.dicts('x_b', [(i, j) for i in I for j in J], lowBound=0)
    s   = pulp.LpVariable.dicts('s',   [(i, j) for i in I for j in J], lowBound=0)
    z_a = pulp.LpVariable.dicts('z_a', [(i, j) for i in I for j in J], lowBound=0)
    z_b = pulp.LpVariable.dicts('z_b', [(i, j) for i in I for j in J], lowBound=0)

    variables = {'x_a': x_a, 'x_b': x_b, 's': s, 'z_a': z_a, 'z_b': z_b}

    # Целевая функция
    p += pulp.lpSum(
        C_BASE[i, j] * (x_a[i, j] + s[i, j] + z_a[i, j]) +
        (C_BASE[i, j] + delta_cost_b) * (x_b[i, j] + z_b[i, j])
        for i in I for j in J
    )

    # Ограничения по запасам на складах
    p += pulp.lpSum(x_a[1, j] + s[1, j] + z_a[1, j] for j in J) <= sup_a1
    p += pulp.lpSum(x_b[1, j] + z_b[1, j] for j in J) <= sup_b1
    p += pulp.lpSum(x_a[2, j] + s[2, j] + z_a[2, j] for j in J) <= sup_a2
    p += pulp.lpSum(x_b[2, j] + z_b[2, j] for j in J) <= sup_b2

    # Ограничения по потребностям
    p += pulp.lpSum(x_a[i, 1] for i in I) == DEMAND[(1, 'a')]
    p += pulp.lpSum(x_b[i, 1] + SUBSTITUTION_RATE * s[i, 1] for i in I) == DEMAND[(1, 'b')]
    p += pulp.lpSum(z_a[i, 1] + z_b[i, 1] for i in I) == DEMAND[(1, 'any')]

    p += pulp.lpSum(x_a[i, 2] for i in I) == DEMAND[(2, 'a')]
    p += pulp.lpSum(x_b[i, 2] + SUBSTITUTION_RATE * s[i, 2] for i in I) == DEMAND[(2, 'b')]
    p += pulp.lpSum(z_a[i, 2] + z_b[i, 2] for i in I) == DEMAND[(2, 'any')]

    if solve:
        p.solve(pulp.PULP_CBC_CMD(msg=False))

    return p, variables


prob, vars_base = create_and_solve_steel_problem(
    SUPPLY_A1_A, SUPPLY_A1_B, SUPPLY_A2_A, SUPPLY_A2_B
)
x_a = vars_base['x_a']
x_b = vars_base['x_b']
s = vars_base['s']
z_a = vars_base['z_a']
z_b = vars_base['z_b']

## Решение задачи. Интерпретация решения в терминах постановки задачи.

In [2]:
print('Статус решения:', pulp.LpStatus[prob.status])
print('Минимальные суммарные затраты: {:.2f}'.format(pulp.value(prob.objective)))

print('Перевозки:')
for i in I:
    for j in J:
        for name, var in [('x_a', x_a), ('x_b', x_b), ('s', s), ('z_a', z_a), ('z_b', z_b)]:
            v = var[(i,j)].value()
            if v is not None and v > 1e-6:
                print('A{} -> B{} {:6s}: {:.2f}'.format(i, j, name, v))

for i in I:
    tot_a = sum((x_a[i,j].value() or 0) + (s[i,j].value() or 0) + (z_a[i,j].value() or 0) for j in J)
    tot_b = sum((x_b[i,j].value() or 0) + (z_b[i,j].value() or 0) for j in J)
    print('Всего с A{} отправлено: марка "а": {:.2f} т, марка "б": {:.2f} т'.format(i, tot_a, tot_b))



Статус решения: Optimal
Минимальные суммарные затраты: 41700.00
Перевозки:
A1 -> B1 x_a   : 2500.00
A1 -> B1 x_b   : 3800.00
A1 -> B1 z_a   : 400.00
A1 -> B2 x_b   : 100.00
A2 -> B2 x_a   : 2700.00
A2 -> B2 x_b   : 2300.00
A2 -> B2 z_a   : 400.00
Всего с A1 отправлено: марка "а": 2900.00 т, марка "б": 3900.00 т
Всего с A2 отправлено: марка "а": 3100.00 т, марка "б": 2300.00 т


## Исследование решения для ответа на вопросы, указанные в варианте задания.

In [3]:
def run_and_analyze(sup_a1, sup_b1, sup_a2, sup_b2, delta_cost_b=0.0):
    p, variables = create_and_solve_steel_problem(sup_a1, sup_b1, sup_a2, sup_b2, delta_cost_b)
    s_variables = variables['s']
    total_substitution = sum(s_variables[i, j].value() or 0 for i in I for j in J)
    return pulp.value(p.objective), total_substitution, s_variables

print("1. Провести параметрическое изменение правых частей")

step = 0
while True:
    a1_a, a1_b = SUPPLY_A1_A + step, SUPPLY_A1_B - step
    a2_a, a2_b = SUPPLY_A2_A + step, SUPPLY_A2_B - step

    if a1_b < 0 or a2_b < 0:
        print(f"| Сдвиг {step} т. | Исследование остановлено: запасы марки 'б' исчерпаны.")
        break

    obj, sub_vol, s_vars = run_and_analyze(a1_a, a1_b, a2_a, a2_b)

    if sub_vol > 0.01:
        print(f"Шаг сдвига запасов: {step} т. Всего заменено стали марки 'а': {sub_vol:.2f} т.")
        for (i, j), var in s_vars.items():
            if var.value() > 1e-5:
                print(f"  -> Направлено на замену по маршруту А{i} -> B{j}: {var.value():.2f} т.")

    if sub_vol >= 100.0:
        print(f"При изменении запасов на {step} т. объем замены составил >= 100 т.")
        break
    step += 5

print("\n2. Провести параметрическое изменение целевой функции")
# 1. Существенно увеличиваем запасы, чтобы они не лимитировали модель
big_stock = 15000
print(f"Увеличиваем запасы всех марок на всех складах до {big_stock} т.")

# 2. Убеждаемся в исходном отсутствии замен при избытке ресурсов
_, init_sub, _ = run_and_analyze(big_stock, big_stock, big_stock, big_stock, delta_cost_b=0)
print(f"Объем замен при базовых ценах и избыточных запасах: {init_sub:.2f} т.")

# 3. Постепенно увеличиваем стоимость перевозки стали «б» (delta_cost_b)
delta_c = 0.0
while True:
    obj, sub_vol, s_vars = run_and_analyze(big_stock, big_stock, big_stock, big_stock, delta_cost_b=delta_c)

    if sub_vol > 0.01:
        print(f"Надбавка к стоимости марки 'б': {delta_c:.2f} ДЕ. Всего заменено: {sub_vol:.2f} т.")

    if sub_vol >= 100.0:
        print(f"При удорожании марки 'б' на {delta_c:.2f} ДЕ замена превысила 100 т.")
        break
    delta_c += 0.05

1. Провести параметрическое изменение правых частей
Шаг сдвига запасов: 405 т. Всего заменено стали марки 'а': 14.29 т.
  -> Направлено на замену по маршруту А1 -> B1: 14.29 т.
Шаг сдвига запасов: 410 т. Всего заменено стали марки 'а': 28.57 т.
  -> Направлено на замену по маршруту А1 -> B1: 28.57 т.
Шаг сдвига запасов: 415 т. Всего заменено стали марки 'а': 42.86 т.
  -> Направлено на замену по маршруту А1 -> B1: 42.86 т.
Шаг сдвига запасов: 420 т. Всего заменено стали марки 'а': 57.14 т.
  -> Направлено на замену по маршруту А1 -> B1: 57.14 т.
Шаг сдвига запасов: 425 т. Всего заменено стали марки 'а': 71.43 т.
  -> Направлено на замену по маршруту А1 -> B1: 71.43 т.
Шаг сдвига запасов: 430 т. Всего заменено стали марки 'а': 85.71 т.
  -> Направлено на замену по маршруту А1 -> B1: 85.71 т.
Шаг сдвига запасов: 435 т. Всего заменено стали марки 'а': 100.00 т.
  -> Направлено на замену по маршруту А1 -> B1: 100.00 т.
При изменении запасов на 435 т. объем замены составил >= 100 т.

2. Про